In [2]:
# pip install xarray netCDF4 pandas numpy scikit-learn xgboost joblib

import numpy as np
import pandas as pd
import xarray as xr
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error
import joblib


In [3]:
import glob
import xarray as xr

PATH = "/Users/lb962/Documents/GitHub/ESL/data/ml_ready/*.nc"

# Get sorted list of files
files = sorted(glob.glob(PATH))

# Open each file into a list of datasets
datasets = [xr.open_dataset(f) for f in files]

# Example: inspect or process one
for i, ds in enumerate(datasets):
    print(f"File {i}: {files[i]}")
    print(ds)

# If you want them in memory:
datasets = [ds.load() for ds in datasets]


File 0: /Users/lb962/Documents/GitHub/ESL/data/ml_ready/small_ds.nc
<xarray.Dataset>
Dimensions:            (station: 9, valid_time: 750192)
Coordinates:
  * station            (station) int64 1 2 3 4 5 6 7 8 10
  * valid_time         (valid_time) datetime64[ns] 1940-01-01 ... 2025-07-30T...
    latitude           float64 ...
    longitude          float64 ...
    station_latitude   (station) float64 ...
    station_longitude  (station) float64 ...
Data variables: (12/14)
    u10                (station, valid_time) float32 ...
    v10                (station, valid_time) float32 ...
    d2m                (station, valid_time) float32 ...
    t2m                (station, valid_time) float32 ...
    msl                (station, valid_time) float32 ...
    sst                (station, valid_time) float32 ...
    ...                 ...
    mwd                (station, valid_time) float32 ...
    mwp                (station, valid_time) float32 ...
    swh                (station, valid_

In [4]:
def preprocess(ds):
    return ds.sortby("station") 

datasets = [preprocess(xr.open_dataset(f)) for f in files]
ds = xr.concat(datasets, dim="station")

In [5]:
ds_recent = ds.where(ds["valid_time"].dt.year >= 2000, drop=True)

In [6]:
from numba import njit
import numpy as np
import xarray as xr
from numba import njit

@njit(cache=True, fastmath=True)
def _mask_1d(values, k_eff, mean, std, safe_lo, safe_hi,
             hard_clip_eff, p95, p95_margin_eff,
             q1, q3, use_quantiles, iqr_mult):
    n = values.shape[0]
    dev = np.zeros(n, dtype=np.bool_)
    last_kept = np.nan; have_last=False; prev_was_kept=False
    z_enabled = np.isfinite(std) and std > 0.0
    p95_gate = p95 + p95_margin_eff if np.isfinite(p95) else np.inf
    iqr = q3 - q1
    lo_iqr = q1 - iqr_mult * iqr
    hi_iqr = q3 + iqr_mult * iqr
    for i in range(n):
        x = values[i]
        if not np.isfinite(x): dev[i]=True; prev_was_kept=False; continue
        if np.abs(x) > hard_clip_eff: dev[i]=True; prev_was_kept=False; continue
        if (x >= safe_lo) and (x <= safe_hi): dev[i]=False; last_kept=x; have_last=True; prev_was_kept=True; continue
        if use_quantiles and ((x < lo_iqr) or (x > hi_iqr)): dev[i]=True; prev_was_kept=False; continue
        if z_enabled:
            z = (x - mean) / std
            if np.abs(z) > k_eff: dev[i]=True; prev_was_kept=False; continue
        if not prev_was_kept:
            if x <= p95_gate: dev[i]=False; last_kept=x; have_last=True; prev_was_kept=True
            else: dev[i]=True; prev_was_kept=False
            continue
        if have_last:
            allowed = 1.5 if (-1.0 <= last_kept <= 1.0) else 0.5
            if np.abs(x - last_kept) > allowed: dev[i]=True; prev_was_kept=False; continue
        dev[i]=False; last_kept=x; have_last=True; prev_was_kept=True
    return dev

# FAST block-drift mask: EW-quantile (median) + run-length
@njit(cache=True, fastmath=True)
def _block_mask_1d(values, med_thresh, min_run, alpha):
    n = values.shape[0]
    block = np.zeros(n, dtype=np.bool_)
    m = 0.0                        # EW median estimate (starts near 0 for residuals)
    run = 0                        # consecutive samples with |m| > threshold
    for i in range(n):
        x = values[i]
        if np.isfinite(x):
            # Robbins–Monro quantile update (tau=0.5 -> median)
            m += alpha * (0.5 - (1.0 if x < m else 0.0))
            if np.abs(m) > med_thresh:
                run += 1
                if run == min_run:
                    # backfill the initial block once threshold run is reached
                    start = i - min_run + 1
                    for j in range(start, i + 1):
                        block[j] = True
                elif run > min_run:
                    block[i] = True
            else:
                run = 0
        else:
            run = 0  # NaN breaks the block
    return block
@njit(cache=True, fastmath=True)
def _yrmean_block_mask_1d(values, yr_mean_vals, good_year_vals,
                          mean_abs_thresh, band, min_run):
    """
    Flag contiguous runs that stay near the (shifted) yearly mean, but only
    in years where |yearly mean| > mean_abs_thresh and coverage was 'good'.
    """
    n = values.shape[0]
    block = np.zeros(n, dtype=np.bool_)
    run = 0
    start = 0
    for i in range(n):
        x = values[i]
        if not np.isfinite(x):
            run = 0
            continue

        if not good_year_vals[i]:
            # year not eligible: reset run
            run = 0
            continue

        ymean = yr_mean_vals[i]
        if not np.isfinite(ymean) or (np.abs(ymean) <= mean_abs_thresh):
            # yearly mean not "shifted" enough
            run = 0
            continue

        # Are we "near" the shifted yearly mean?
        if np.abs(x - ymean) <= band:
            if run == 0:
                start = i
            run += 1
            if run == min_run:
                for j in range(start, i + 1):
                    block[j] = True
            elif run > min_run:
                block[i] = True
        else:
            run = 0
    return block
def filter_residual(
    ds: xr.Dataset,
    var: str = "residual",
    dim: str | None = None,
    base_k: float = 5.0,
    safe_keep_band: tuple[float, float] = (-2.0, 2.0),
    hard_clip: float = 3.0,
    p95_margin: float = 0.5,
    station_dim: str = "station",
    min_frac_year: float = 0.5,
    kurt_hard: float = 1.0,
    # mean-shift block knobs
    year_mean_abs_thresh: float = 1.0,   # consider a year "shifted" if |year mean| > 1 m
    block_band: float = 0.5,             # samples within ±band of the yearly mean count toward a block
    min_run: int = 60,                   # consecutive samples near the mean to declare a block
):
    da = ds[var]; dim = da.dims[0] if dim is None else dim
    ds = ds.sortby(dim); da = ds[var].astype("float64")

    # --- robust stats (unchanged)
    mu   = da.mean(dim=dim, skipna=True)
    sig  = da.std(dim=dim,  skipna=True)
    p95  = da.quantile(0.95, dim=dim, skipna=True)
    qs   = da.quantile([0.25, 0.75], dim=dim, skipna=True)
    q1, q3 = qs.sel(quantile=0.25), qs.sel(quantile=0.75)

    m2 = da.var(dim=dim, skipna=True)
    m4 = ((da - mu) ** 4).mean(dim=dim, skipna=True)
    ex_kurt = xr.where(m2 > 0, m4 / (m2 ** 2) - 3.0, 0.0)
    heavy = ex_kurt > kurt_hard

    k_eff          = xr.where(heavy, base_k / 2.5, base_k)
    iqr_mult       = xr.where(heavy, 1.0, 1.5)
    hard_clip_eff  = xr.where(heavy, 2.0, hard_clip)
    p95_margin_eff = xr.where(heavy, 0.1, p95_margin)
    use_quantiles  = heavy

    dev_point = xr.apply_ufunc(
        _mask_1d,
        da, k_eff, mu, sig,
        xr.DataArray(safe_keep_band[0]), xr.DataArray(safe_keep_band[1]),
        hard_clip_eff, p95, p95_margin_eff,
        q1, q3, use_quantiles, iqr_mult,
        input_core_dims=[[dim], [], [], [], [], [], [], [], [], [], [], [], []],
        output_core_dims=[[dim]],
        vectorize=True, dask="parallelized",
        output_dtypes=[bool],
    )

    # --- yearly mean and coverage (unchanged)
    present = da.notnull()
    cov_year = present.groupby(f"{dim}.year").mean(dim=dim)         # fraction present per year
    yr_mean  = da.groupby(f"{dim}.year").mean(dim=dim, skipna=True)  # mean per year
    good_year = cov_year > min_frac_year

    # --- NEW: broadcast per-year values back to each timestamp by selecting with a per-sample 'year' indexer
    # Ensure time axis is datetime64
    if not np.issubdtype(ds[dim].dtype, np.datetime64):
        # If your coord isn't datetime64, convert it here (one-time)
        ds = ds.assign_coords({dim: xr.converters.to_datetime64(ds[dim])})
        da = ds[var].astype("float64")

    year_indexer = xr.DataArray(
        ds[dim].dt.year,         # vector of years for every sample along `dim`
        dims=dim,
        coords={dim: ds[dim]}
    )

    # These now have the SAME dims as `da` (broadcasted back from 'year')
    yr_mean_full   = yr_mean.sel(year=year_indexer)
    good_year_full = good_year.sel(year=year_indexer)

    # --- block detector: runs near the shifted yearly mean
    dev_block = xr.apply_ufunc(
        _yrmean_block_mask_1d,
        da,
        yr_mean_full,
        good_year_full,
        xr.DataArray(year_mean_abs_thresh),
        xr.DataArray(block_band),
        xr.DataArray(np.int64(min_run)),
        input_core_dims=[[dim], [dim], [dim], [], [], []],
        output_core_dims=[[dim]],
        vectorize=True, dask="parallelized",
        output_dtypes=[bool],
    )

    deviant = dev_point | dev_block

    out = ds.copy()
    out[f"{var}_is_deviant"] = deviant
    out[var] = da.where(~deviant)

    if station_dim in out.dims:
        extra = [d for d in cov_year.dims if d not in (station_dim, "year")]
        cov_check = cov_year if not extra else cov_year.mean(dim=extra)
        keep_station = (cov_check > min_frac_year).any(dim="year")
        out = out.isel({station_dim: keep_station})

    return out, out[f"{var}_is_deviant"]
# ds: xr.Dataset with a variable named "residual" along dimension "time"
ds_filtered, dev_mask = filter_residual(ds, var="residual", dim="valid_time")
ds = ds_filtered

/Users/lb962/miniconda3/envs/ESL/lib/python3.12/site-packages/numpy/lib/nanfunctions.py:1563: RuntimeWarning: All-NaN slice encountered
  return function_base._ureduce(a,


In [7]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd
import xarray as xr

# 1) Annual maxima of sea_level per station
annual_max = ds["residual"].groupby("valid_time.year").max("valid_time", skipna=True)  # (station, year)

# 2) Require at least 10 annual maxima to estimate a 10-yr RL
n_years = annual_max.notnull().sum("year")                                              # (station,)
tenyr_rl = annual_max.quantile(0.9, dim="year", skipna=True)                             # (station,)
tenyr_rl = tenyr_rl.where(n_years >= 3)                                                # mask weak stations

# 3) Pull station coords (1D over 'station')
lons = ds["station_longitude"].values
lats = ds["station_latitude"].values
vals = tenyr_rl.values

# Reuse coords & units from your code
proj = ccrs.PlateCarree()
valid_pts = np.isfinite(lons) & np.isfinite(lats)

def quick_map(values, title, cbarlabel="", cmap="plasma_r", vmin=None, vmax=None, n_clusters=None):
    vals = np.asarray(values)
    proj = ccrs.PlateCarree()
    fig = plt.figure(figsize=(10,8))
    ax = plt.axes(projection=proj)

    # Compute valid extent
    valid_pts = np.isfinite(lons) & np.isfinite(lats)
    xmin, xmax = float(np.nanmin(lons[valid_pts])), float(np.nanmax(lons[valid_pts]))
    ymin, ymax = float(np.nanmin(lats[valid_pts])), float(np.nanmax(lats[valid_pts]))
    pad_x = max(0.5, 0.05 * (xmax - xmin))
    pad_y = max(0.5, 0.05 * (ymax - ymin))
    ax.set_extent([xmin - pad_x, xmax + pad_x, ymin - pad_y, ymax + pad_y], crs=proj)

    # Base map features
    ax.add_feature(cfeature.LAND, facecolor="lightgray")
    ax.add_feature(cfeature.OCEAN, facecolor="whitesmoke")
    ax.add_feature(cfeature.COASTLINE, linewidth=0.7)
    ax.add_feature(cfeature.BORDERS, linewidth=0.5, linestyle=":")
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, color="gray", alpha=0.5, linestyle="--")
    gl.right_labels = False; gl.top_labels = False

    # --- 🔹 Make discrete colorbar if n_clusters is given ---
    if n_clusters is not None and n_clusters > 1:
        cmap_obj = plt.get_cmap(cmap, n_clusters)
        norm = mcolors.BoundaryNorm(np.linspace(vmin, vmax, n_clusters + 1), cmap_obj.N)
    else:
        cmap_obj = plt.get_cmap(cmap)
        norm = mcolors.Normalize(vmin=vmin, vmax=vmax)

    # Scatter plot
    sc = ax.scatter(lons, lats, c=vals, s=150, cmap=cmap_obj, norm=norm,
                    edgecolor="black", linewidth=0.4, alpha=0.95, transform=proj)

    # Discrete colorbar
    cb = plt.colorbar(sc, ax=ax, fraction=0.036, pad=0.02)
    cb.set_label(cbarlabel)
    if n_clusters is not None and n_clusters > 1:
        cb.set_ticks(np.linspace(vmin, vmax, n_clusters))
        cb.ax.set_yticklabels([f"C{i+1}" for i in range(n_clusters)])  # optional: label by cluster number

    ax.set_title(title)
    plt.tight_layout()
    plt.show()


In [ ]:
# ================= Q–Q fit vs lognormal, gamma, GEV (per station) =================
import numpy as np
import xarray as xr
import scipy.stats as st
from numpy.linalg import lstsq
import warnings

# ---- config ---------------------------------------------------------------------
VAR_NAME = "residual"     # data variable to evaluate
MIN_N = 20                # min finite samples required to attempt a fit
PROBS = np.linspace(0.01, 0.99, 99)  # quantile levels for Q–Q
DISTRIBUTIONS = [
    ("lognorm", st.lognorm, {"require_pos": True}),
    ("gamma",   st.gamma,   {"require_pos": True}),
    ("gev",     st.genextreme, {"require_pos": False}),  # scipy paramization of GEV
]

# ---- optional: a cartopy map helper if you don't want to rely on quick_map -------
USE_QUICK_MAP = True  # set False to force the built-in cartopy plotter below

import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.colors import ListedColormap, BoundaryNorm

def map_values_cartopy(lon, lat, vals, title="", cbarlabel="", vmin=None, vmax=None,
                       categorical=False, classes=None, class_labels=None):
    proj = ccrs.Robinson()
    data_crs = ccrs.PlateCarree()
    valid = np.isfinite(lon) & np.isfinite(lat) & np.isfinite(vals)
    if valid.sum() == 0:
        raise ValueError("No valid points to plot.")

    fig = plt.figure(figsize=(12, 6))
    ax = plt.axes(projection=proj)
    ax.set_global()
    ax.coastlines(linewidth=0.6)
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), linewidth=0.3)
    ax.add_feature(cfeature.LAND.with_scale("50m"), facecolor="0.95")
    ax.add_feature(cfeature.OCEAN.with_scale("50m"), facecolor="white")

    if categorical:
        if classes is None:
            classes = np.unique(vals[valid].astype(int))
        n = len(classes)
        cmap = ListedColormap(plt.get_cmap("tab20").colors[:n])
        # remap classes to 0..n-1
        class_to_i = {c:i for i,c in enumerate(classes)}
        idx = np.full(vals.shape, np.nan)
        for c,i in class_to_i.items():
            idx[vals == c] = i
        sc = ax.scatter(lon[valid], lat[valid], c=idx[valid],
                        transform=data_crs, s=70, cmap=cmap,
                        edgecolor="k", linewidth=0.25)
        # legend
        labels = class_labels if class_labels is not None else [str(c) for c in classes]
        handles = [plt.Line2D([0],[0], marker='o', linestyle='',
                   markerfacecolor=cmap(i), markeredgecolor='k',
                   label=labels[j]) for j,i in enumerate(range(n))]
        ax.legend(handles=handles, loc="upper left", frameon=False, ncol=min(5, n))
    else:
        sc = ax.scatter(lon[valid], lat[valid], c=vals[valid],
                        transform=data_crs, s=70, cmap="viridis",
                        vmin=vmin, vmax=vmax,
                        edgecolor="k", linewidth=0.25)
        cb = plt.colorbar(sc, orientation="horizontal", pad=0.04, shrink=0.8)
        cb.set_label(cbarlabel)
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

# ---- helpers to compute Q–Q metrics -------------------------------------------
def qq_metrics(x, dist, params, probs=PROBS):
    """
    Given data x and (dist, params), build Q–Q points and fit y ~ a*x + b:
        y = empirical quantiles, x = theoretical quantiles
    Returns (slope, intercept, R2, rmse).
    """
    # empirical quantiles
    y = np.quantile(x, probs)
    # theoretical quantiles from fitted dist
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        x_theor = dist.ppf(probs, *params)

    # filter finite pairs
    m = np.isfinite(x_theor) & np.isfinite(y)
    if m.sum() < 3:
        return (np.nan, np.nan, np.nan, np.nan)

    X = np.column_stack([x_theor[m], np.ones(m.sum())])
    a, b = lstsq(X, y[m], rcond=None)[0]  # slope, intercept
    yhat = a * x_theor[m] + b
    # R2
    ss_res = np.sum((y[m] - yhat) ** 2)
    ss_tot = np.sum((y[m] - y[m].mean()) ** 2)
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan
    rmse = np.sqrt(np.mean((y[m] - yhat) ** 2))
    return (a, b, r2, rmse)

def fit_params(dist_name, dist_obj, x):
    """Fit distribution parameters safely, with domain guards."""
    # guard for positive-only distributions
    if dist_name in ("lognorm", "gamma"):
        x = x[x > 0]
        if x.size < MIN_N:
            return None, None

    if x.size < MIN_N:
        return None, None

    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            params = dist_obj.fit(x)  # MLE (shape(s), loc, scale)
        return x, params
    except Exception:
        return None, None

# ---- iterate over stations -----------------------------------------------------
n_stations = ds.sizes["station"]
results = {
    # per distribution arrays (station,)
    "lognorm_r2": np.full(n_stations, np.nan),
    "gamma_r2":   np.full(n_stations, np.nan),
    "gev_r2":     np.full(n_stations, np.nan),
    "lognorm_rmse": np.full(n_stations, np.nan),
    "gamma_rmse":   np.full(n_stations, np.nan),
    "gev_rmse":     np.full(n_stations, np.nan),
    # store a tiny flag: best distribution by R2 (0=lognorm, 1=gamma, 2=GEV)
    "best_dist":  np.full(n_stations, np.nan),
}

with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=RuntimeWarning)

    for i in range(n_stations):
        x = ds[VAR_NAME].isel(station=i).values
        x = x[np.isfinite(x)]
        if x.size < MIN_N:
            continue

        r2s = []
        rmses = []
        for di, (name, dist_obj, meta) in enumerate(DISTRIBUTIONS):
            x_fit, params = fit_params(name, dist_obj, x)
            if x_fit is None:
                r2s.append(np.nan); rmses.append(np.nan)
                continue
            slope, intercept, r2, rmse = qq_metrics(x_fit, dist_obj, params, probs=PROBS)
            results[f"{name}_r2"][i] = r2
            results[f"{name}_rmse"][i] = rmse
            r2s.append(r2); rmses.append(rmse)

        # pick best by R²
        if np.isfinite(r2s).any():
            best = int(np.nanargmax(r2s))
            results["best_dist"][i] = best

# ---- plots ---------------------------------------------------------------------
# Get coordinates (use your robust getter; pass explicit names if needed)
lon, lat = get_station_coords(ds)  # or get_station_coords(ds, lon_name="LON", lat_name="LAT")

# 1) Best distribution (categorical)
best_vals = results["best_dist"]
labels = ["lognorm", "gamma", "GEV"]
if USE_QUICK_MAP:
    # quick_map doesn't do categorical, so show as 0/1/2 index with a legend-based map
    # use our helper for categorical display
    map_values_cartopy(lon, lat, best_vals, title="Best-fit distribution by Q–Q (max R²)",
                       categorical=True, classes=np.array([0,1,2]),
                       class_labels=labels)
else:
    map_values_cartopy(lon, lat, best_vals, title="Best-fit distribution by Q–Q (max R²)",
                       categorical=True, classes=np.array([0,1,2]),
                       class_labels=labels)

# 2) Continuous R² maps (one per distribution)
for name in ["lognorm", "gamma", "gev"]:
    vals = results[f"{name}_r2"]
    title = f"Q–Q R² vs {name}"
    cbar = "Q–Q R²"
    if USE_QUICK_MAP:
        # Clamp display between 0 and 1
        vals_plot = vals.copy()
        # quick_map signature: quick_map(vals, title, cbarlabel)
        try:
            quick_map(vals_plot, title, cbar)
        except Exception:
            # fallback to explicit cartopy function
            map_values_cartopy(lon, lat, vals, title=title, cbarlabel=cbar, vmin=0.0, vmax=1.0)
    else:
        map_values_cartopy(lon, lat, vals, title=title, cbarlabel=cbar, vmin=0.0, vmax=1.0)

# Optional: make a tidy DataFrame of metrics to save/inspect
qq_df = pd.DataFrame({
    "lognorm_r2": results["lognorm_r2"],
    "gamma_r2":   results["gamma_r2"],
    "gev_r2":     results["gev_r2"],
    "lognorm_rmse": results["lognorm_rmse"],
    "gamma_rmse":   results["gamma_rmse"],
    "gev_rmse":     results["gev_rmse"],
    "best_dist_idx": results["best_dist"],
    "best_dist_label": [labels[int(i)] if np.isfinite(i) else np.nan for i in results["best_dist"]],
})
qq_df.insert(0, "station_index", np.arange(n_stations))
print(qq_df.head())
